# 向前预测与概率校准

将训练、校准和最终评价分开。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

生成有时间顺序的二元标签；前期训练逻辑模型。

In [ ]:
from scipy.special import expit
from scipy.optimize import minimize
n=900;x=rng.normal(size=n);truth=expit(-.3+1.2*x);y=rng.binomial(1,truth)
X=np.column_stack([np.ones(n),x]);train=np.arange(300);cal=np.arange(300,600);test=np.arange(600,900)
def logloss(beta,design,label):
    score=design@beta;return np.mean(np.logaddexp(0,score)-label*score)
fit=minimize(logloss,np.zeros(2),args=(X[train],y[train]));score=X@fit.x
raw=expit(2*score) # deliberately overconfident probabilities
print('model coefficients:',fit.x)

校准器只拟合中期分数，在更晚样本计算 Brier。

In [ ]:
cal_X=np.column_stack([np.ones(n),2*score]);cal_fit=minimize(logloss,np.array([0.,1.]),args=(cal_X[cal],y[cal]))
calibrated=expit(cal_X@cal_fit.x)
print('test Brier raw, calibrated:',np.mean((raw[test]-y[test])**2),np.mean((calibrated[test]-y[test])**2))
for lower in [0,.2,.4,.6,.8]:
    group=test[(calibrated[test]>=lower)&(calibrated[test]<lower+.2)]
    if len(group):print('bin, count, probability, frequency:',lower,len(group),calibrated[group].mean(),y[group].mean())

## 自己试一试

换随机种子，校准是否每次都改善？

## 反馈

不保证。有限样本校准器也有估计误差；应在独立后期评价，并观察样本量与分箱频数。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。